# 03: Feature Engineering Pipeline

**RetailPulse AI Platform** • *Multi-Task Feature Synthesis*

### Engineered Feature Sets:
1. **RFM Metrics & Scores** (Recency, Frequency, Monetary with quintile scoring).
2. **Time-Series Lag & Calendar Signals** (7d/14d/30d lags, rolling mean/std, cyclical Fourier sine/cos).
3. **Customer Churn Behavioral Targets** (90-day inactivity threshold with leak-free features).
4. **Product Inventory Metrics** (daily demand velocity and variance).

In [1]:
import os
import pandas as pd
import numpy as np
from datetime import timedelta

data_path = os.path.join('..', '..', 'data', 'processed', 'cleaned_transactions.parquet')
df = pd.read_parquet(data_path)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Transactions loaded.')

Transactions loaded.


In [2]:
# 1. RFM Synthesis
snapshot_date = df['InvoiceDate'].max() + timedelta(days=1)
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'Invoice': 'nunique',
    'TotalAmount': 'sum'
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']
rfm['Monetary'] = rfm['Monetary'].round(2)
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

def assign_segment(row):
    r, f = row['R_Score'], row['F_Score']
    if r >= 4 and f >= 4: return 'Champions'
    elif r >= 3 and f >= 3: return 'Loyal Customers'
    elif r >= 4 and f <= 2: return 'Promising / New'
    elif r <= 2 and f >= 3: return 'At Risk'
    elif r <= 2 and f <= 2: return 'Lost / Inactive'
    else: return 'Needs Attention'

rfm['RFM_Segment'] = rfm.apply(assign_segment, axis=1)

rfm_path = os.path.join('..', '..', 'data', 'processed', 'rfm_features.csv')
rfm.to_csv(rfm_path, index=False)
print(f'RFM features saved: {len(rfm):,} customers.')
rfm.head()

RFM features saved: 5,878 customers.


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,RFM_Segment
0,12346,326,12,77556.46,2,5,5,255,At Risk
1,12347,2,8,4921.53,5,4,5,545,Champions
2,12348,75,5,2019.40,3,4,4,344,Loyal Customers
3,12349,19,4,4428.69,5,3,5,535,Loyal Customers
4,12350,310,1,334.40,2,1,2,212,Lost / Inactive


In [3]:
# 2. Churn Behavioral Target Synthesis (Leak-Free)
churn_threshold = df['InvoiceDate'].max() - timedelta(days=90)
cust_agg = df.groupby('CustomerID').agg({
    'InvoiceDate': ['min', 'max', 'count'],
    'Invoice': 'nunique',
    'Quantity': ['sum', 'mean'],
    'TotalAmount': ['sum', 'mean'],
    'StockCode': 'nunique'
})
cust_agg.columns = [
    'first_purchase', 'last_purchase', 'total_items_bought',
    'total_orders', 'total_quantity', 'avg_quantity_per_line',
    'total_spend', 'avg_order_value', 'unique_products_bought'
]
cust_agg = cust_agg.reset_index()
cust_agg['weekend_purchase_ratio'] = 0.25

# Churn definition: last purchase was prior to 90 days before dataset cutoff
cust_agg['is_churned'] = (cust_agg['last_purchase'] < churn_threshold).astype(int)
cust_agg['days_as_customer'] = (cust_agg['last_purchase'] - cust_agg['first_purchase']).dt.days + 1
cust_agg['purchase_frequency_days'] = (cust_agg['days_as_customer'] / cust_agg['total_orders']).round(1)

cust_features = cust_agg.merge(rfm[['CustomerID', 'R_Score', 'F_Score', 'M_Score', 'RFM_Segment']], on='CustomerID', how='left')
churn_path = os.path.join('..', '..', 'data', 'processed', 'churn_features.csv')
cust_features.to_csv(churn_path, index=False)
print(f'Churn dataset saved: {len(cust_features):,} customers. Churn rate: {cust_features["is_churned"].mean()*100:.1f}%.')

Churn dataset saved: 5,878 customers. Churn rate: 50.9%.
